# 4회차 실습 — Codex 활용 AI Agent 서비스 개발

## 학습목표
- 3회차 Agent 코드를 서비스 프로젝트 구조로 분리한다.
- FastAPI로 `/health`, `/tools`, `/agent` API를 구성한다.
- Codex를 이용해 기존 코드를 분석하고 수정·확장한다.
- API 문서(Swagger UI)에서 Agent 서비스를 테스트한다.
- 변경 전후 코드를 검토하고 GitHub에 기록한다.

### 실행 구분
- 🟢 LOCAL: 로컬 PC에서 코드 작성·Codex 수정 가능
- 🔴 GPU: Qwen3-8B 실제 Agent 추론
- 📸 CAPTURE: 강의자료 캡처 권장

> 4회차 핵심: **Notebook 속 Agent → 실행 가능한 서비스 코드**


## Step 1. Starter Project 구조 확인 — 🟢 LOCAL

```text
06_agent_service_starter/
├─ app/
│  ├─ tools.py      # Custom Tool
│  ├─ llm.py        # Qwen 로딩·추론
│  ├─ agent.py      # Agent 판단·Tool 실행
│  └─ main.py       # FastAPI
├─ requirements.txt
└─ README.md
```

3회차 코드를 기능별 파일로 분리하여 **수정·테스트·확장하기 쉬운 구조**로 바꿉니다.


In [ ]:
from pathlib import Path

project_dir = Path("06_agent_service_starter")
for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path)


## Step 2. Codex에 프로젝트 폴더 열기 — 🟢 LOCAL / 📸

현재 ChatGPT 데스크톱 앱의 **Codex**에서 Starter Project 폴더를 엽니다.

### 첫 번째 프롬프트
```text
이 프로젝트는 Qwen3-8B와 Python Custom Tool을 사용하는 교육용 AI Agent 서비스다.
먼저 코드를 수정하지 말고 프로젝트 구조와 각 파일의 역할을 분석해줘.
특히 tools.py → agent.py → main.py로 요청이 흐르는 과정을 설명해줘.
```

**중요:** 처음부터 “전부 고쳐줘”라고 하지 말고, 먼저 구조를 분석하게 합니다.


## Step 3. 로컬에서 Tool 코드 확인 — 🟢 LOCAL


In [ ]:
# Starter Project를 현재 Notebook과 같은 폴더에 풀었다고 가정합니다.
tools_path = project_dir / "app" / "tools.py"
print(tools_path.read_text(encoding="utf-8")[:4000])


## Step 4. FastAPI의 역할 이해 — 🟢 LOCAL

FastAPI는 Agent 함수를 HTTP API로 노출합니다.

- `GET /health` : 서비스와 GPU 상태 확인
- `GET /tools` : 사용 가능한 Tool 목록
- `POST /agent` : 사용자 질문을 Agent에 전달


In [ ]:
main_path = project_dir / "app" / "main.py"
print(main_path.read_text(encoding="utf-8"))


## Step 5. Codex 미션 1 — 코드 설명과 개선점 찾기 — 🟢 LOCAL

### 프롬프트
```text
app/main.py, app/agent.py, app/tools.py를 검토해줘.
교육용 Starter Code라는 점을 고려해서
1) 현재 구조의 장점
2) 오류가 날 수 있는 부분
3) 학생이 이해하기 어려운 부분
을 구분해서 설명해줘.
아직 코드는 수정하지 마.
```

학생은 Codex의 분석을 그대로 수용하지 않고 **수정할 항목을 직접 선택**합니다.


## Step 6. Codex 미션 2 — 새로운 Tool 추가 — 🟢 LOCAL

예: `word_count(text)` Tool을 추가합니다.

### 프롬프트
```text
app/tools.py에 word_count(text) Tool을 추가해줘.
문자열을 공백 기준으로 나누어 단어 수를 반환하고,
기존 Tool과 동일하게 {"ok": true, ...} 형태로 반환해줘.
TOOL_REGISTRY와 TOOL_DESCRIPTIONS에도 등록해줘.
기존 기능은 변경하지 마.
```

수정 후 반드시 **diff를 검토**합니다.


## Step 7. 변경 코드 직접 확인 — 🟢 LOCAL / 📸

Codex가 수정한 뒤에는 다음을 확인합니다.

1. 요청한 파일만 수정했는가?
2. 기존 Tool을 삭제하거나 바꾸지 않았는가?
3. 함수 입력·출력 형식이 기존 규칙과 일치하는가?
4. Registry와 Description에 모두 등록되었는가?
5. 위험한 실행 코드가 추가되지 않았는가?


In [ ]:
# Codex 수정 후 다시 실행하여 변경 내용을 확인합니다.
print((project_dir / "app" / "tools.py").read_text(encoding="utf-8")[:5000])


## Step 8. FastAPI 설치 및 import 확인 — 🟢 LOCAL

로컬에서는 GPU 모델을 실제 로드하지 않고도 프로젝트의 기본 문법과 import 구조를 점검할 수 있습니다.

터미널:
```bash
pip install fastapi "uvicorn[standard]" pydantic
python -m compileall app
```


## Step 9. GPU 환경 준비 — 🔴 GPU

RunPod를 시작한 뒤 프로젝트를 GitHub에서 가져오거나 업로드합니다.

```bash
nvidia-smi
cd /root/yu-gpu-ai
git pull
```

프로젝트 폴더로 이동하여 의존성을 설치합니다.

```bash
pip install -r requirements.txt
```


## Step 10. FastAPI Agent Service 실행 — 🔴 GPU

터미널에서:

```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

정상 실행 후 RunPod의 HTTP Service/프록시를 통해 서비스에 접속합니다.

> 실제 공개 URL 형태는 RunPod 환경 설정에 따라 달라질 수 있으므로 수업에서는 현재 Pod 화면에서 확인합니다.


## Step 11. `/health` 테스트 — 🔴 GPU / 📸

예상 확인 항목:
- `status`
- `model`
- `cuda_available`
- `gpu`

Qwen 모델을 실제 호출하기 전에 **서비스와 GPU 상태부터 확인**합니다.


## Step 12. `/tools` 테스트 — 🟢/🔴 / 📸

Swagger UI에서 `GET /tools`를 실행합니다.

확인:
- calculator
- text_length
- current_time
- 학생이 추가한 Custom Tool

Tool Registry가 API에 제대로 노출되는지 확인합니다.


## Step 13. `/agent` 테스트 — 🔴 GPU / 📸

Swagger UI의 `POST /agent`에서:

```json
{
  "query": "33명이 30시간씩 GPU를 사용하면 총 몇 GPU-hour야?"
}
```

확인할 항목:
- `status`
- `used_tool`
- `tool_name`
- `arguments`
- `tool_result`
- `final_answer`


## Step 14. 서비스 요청 흐름 추적 — 📸

```text
POST /agent
   ↓
main.py
   ↓
run_agent()
   ↓
Qwen3-8B 판단
   ↓
execute_tool()
   ↓
Tool Result
   ↓
Qwen3-8B 최종 응답
   ↓
JSON Response
```

학생은 오류가 발생했을 때 **어느 계층에서 문제가 생겼는지** 구분합니다.


## Step 15. Codex 미션 3 — 오류 응답 개선 — 🟢 LOCAL

### 프롬프트
```text
현재 POST /agent에서 Agent 내부 오류가 발생해도 HTTP 응답 구조가 충분히 명확하지 않다.
기존 run_agent()의 반환 구조를 최대한 유지하면서,
main.py에서 status가 error 또는 tool_error일 때
클라이언트가 원인을 이해하기 쉬운 응답을 만들 수 있도록 개선안을 제안해줘.
먼저 변경 계획을 보여주고 내가 확인한 뒤 수정해줘.
```

**계획 → 검토 → 수정** 순서로 Codex를 사용합니다.


## Step 16. Codex 미션 4 — 기능 추가 — 🟢 LOCAL

팀별로 하나를 선택합니다.

- Agent 실행 로그를 JSONL 파일로 저장
- `/tools/{tool_name}` 상세 조회 API 추가
- 요청 ID(request_id) 추가
- Tool 실행 시간 측정
- 간단한 `/stats` API 추가

Codex 프롬프트에는 반드시:
1. 변경 목표
2. 수정 허용 파일
3. 유지해야 할 기존 기능
4. 완료 조건
을 포함합니다.


## Step 17. Codex 프롬프트 템플릿 — 📸

```text
[목표]
Agent 실행 시간을 측정해서 API 응답에 elapsed_ms를 추가한다.

[수정 허용 파일]
app/agent.py

[유지 조건]
기존 status, used_tool, tool_name, final_answer 필드는 유지한다.

[완료 조건]
Tool 사용/미사용 질문 모두 elapsed_ms가 포함되어야 한다.

먼저 변경 계획을 설명하고, 그 다음 코드를 수정해줘.
수정 후 변경 내용을 요약해줘.
```


## Step 18. API 테스트 체크리스트 — 🔴 GPU

| 테스트 | 기대 결과 |
|---|---|
| `/health` | CUDA/GPU 상태 확인 |
| `/tools` | Tool 목록 반환 |
| 계산 질문 | calculator 선택 |
| 글자 수 질문 | text_length 선택 |
| 일반 설명 | Tool 없이 answer |
| 잘못된 요청 | 오류를 구조화해서 반환 |
| 추가 Tool | Registry와 Agent에서 정상 사용 |


## Step 19. Git 변경사항 확인 — 🟢 LOCAL

```bash
git status
git diff
```

확인 후:

```bash
git add .
git commit -m "Build FastAPI agent service"
git push
```

학생 PAT를 교수자나 TA가 수집하지 않습니다.


## Step 20. 4회차 산출물

```text
06_agent_service_starter/
├─ app/
│  ├─ tools.py
│  ├─ llm.py
│  ├─ agent.py
│  └─ main.py
├─ requirements.txt
└─ README.md
```

추가 산출물:
- Codex로 추가/개선한 기능 1개 이상
- Swagger API 테스트 결과
- Git diff 및 commit


## 4회차 실습 완료

### 오늘의 변화
**Notebook 안에서 실행하던 Agent**
→ **파일 구조를 가진 프로그램**
→ **HTTP API로 호출 가능한 Agent Service**

### 다음 차시
Web UI와 FastAPI Agent API를 연결하고, 계층별 디버깅과 성능평가를 통해 **AI Agent Web Service MVP**를 완성합니다.
